# EventLens SPY forecasting experiment

This notebook is organized as five stages: get data, extract features, analyze correlations, fit models, and compare models. It predicts SPY cumulative return and future realized variance over 1, 5, and 20 trading days. Explanations are placed next to the code they describe.


## 1. Get data

We load daily SPY prices/returns and prediction-market snapshots. A forecast at the end of day *t* may use information dated *t* or earlier. The later `final_outcome` is never used as an input because it would reveal the future.


In [1]:
"""EventLens five-stage SPY forecasting experiment.

Stages:
1. Get data
2. Extract features
3. Analyze correlations
4. Fit models
5. Compare models

The script mirrors the annotated notebook. It uses only information available
on or before each forecast date and never uses final_outcome as a feature.
"""

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


# ============================================================================
# 1. GET DATA
# ============================================================================

ROOT = Path.cwd()
if not (ROOT / 'datasets').exists():
    ROOT = ROOT.parent
PROBABILITY_PATH = ROOT / "datasets" / "forward_probabilities.csv"
MARKET_PATH = ROOT / "datasets" / "index_daily.csv"
RESULTS_PATH = ROOT / "models" / "forecast_model_results.csv"
CORRELATIONS_PATH = ROOT / "models" / "feature_return_correlations.csv"
HORIZONS = (1, 5, 20)
SEED = 42

OUTCOME_MOVE = {
    "25 bps decrease": -25.0,
    "50+ bps decrease": -50.0,
    "50 bps decrease": -50.0,
    "75+ bps decrease": -75.0,
    "25 bps increase": 25.0,
    "25+ bps increase": 50.0,
    "50 bps increase": 50.0,
    "50+ bps increase": 50.0,
    "75 bps increase": 75.0,
    "no change": 0.0,
}



The first code cell defines paths, possible rate-move values, and the data-loading function. In a notebook, `ROOT` uses the working directory rather than `__file__`; `__file__` exists in a `.py` script but normally does not exist in a notebook.


In [2]:
def load_data():
    """Load SPY prices and prediction-market snapshots."""
    market = pd.read_csv(MARKET_PATH, parse_dates=["date"])
    market = market.loc[market["ticker"].eq("SPY")].sort_values("date").copy()
    market["return_1d"] = pd.to_numeric(market["return_1d"], errors="coerce")
    market["close"] = pd.to_numeric(market["close"], errors="coerce")
    market = market.dropna(subset=["close", "return_1d"])
    market["date"] = market["date"].dt.tz_localize(None).dt.normalize()

    probability = pd.read_csv(PROBABILITY_PATH, parse_dates=["date"])
    probability["date"] = probability["date"].dt.tz_localize(None).dt.normalize()
    probability["move"] = probability["outcome"].map(OUTCOME_MOVE)
    probability = probability.dropna(subset=["move"]).copy()
    return market, probability



## 2. Extract features

The next cells turn raw probabilities into interpretable features. The selected meeting is the closest meeting available on each date. We do not select rows using `final_outcome`, because that outcome is only known later.


In [3]:
# ============================================================================
# 2. EXTRACT FEATURES
# ============================================================================

def extract_event_features(probability):
    """Create point-in-time features from the nearest available meeting.

    Each date keeps the smallest available horizon. `final_outcome` is not
    used: it is only known after the meeting and would leak future information.
    """
    chosen = (
        probability.sort_values(["date", "horizon_month"])
        .groupby("date", as_index=False)
        .first()
    )
    selected = probability.merge(
        chosen[["date", "horizon_month"]],
        on=["date", "horizon_month"],
        how="inner",
    )
    selected["market_probability"] = pd.to_numeric(
        selected["market_probability"], errors="coerce"
    )
    selected = selected.dropna(subset=["market_probability"])
    selected["outcome_group"] = np.select(
        [
            selected["move"] < 0,
            selected["move"] > 0,
            selected["move"].eq(0),
        ],
        ["decrease", "increase", "no_change"],
        default="other",
    )
    selected["total_probability"] = selected.groupby("date")[
        "market_probability"
    ].transform("sum")
    selected["p"] = selected["market_probability"] / selected["total_probability"]

    # Probability features: each has a different interpretation.
    selected["weighted_move"] = selected["p"] * selected["move"]
    selected["weighted_abs_move"] = selected["p"] * selected["move"].abs()
    selected["weighted_square_move"] = selected["p"] * selected["move"].pow(2)
    selected["entropy_part"] = -selected["p"] * np.log(
        selected["p"].clip(lower=1e-12)
    )

    event = selected.groupby("date", as_index=False).agg(
        horizon_month=("horizon_month", "first"),
        expected_move=("weighted_move", "sum"),
        expected_abs_move=("weighted_abs_move", "sum"),
        expected_square_move=("weighted_square_move", "sum"),
        entropy=("entropy_part", "sum"),
        max_probability=("p", "max"),
        outcome_count=("p", "size"),
        probability_increase=("p", lambda values: values[
            selected.loc[values.index, "outcome_group"].eq("increase")
        ].sum()),
        probability_decrease=("p", lambda values: values[
            selected.loc[values.index, "outcome_group"].eq("decrease")
        ].sum()),
        probability_no_change=("p", lambda values: values[
            selected.loc[values.index, "outcome_group"].eq("no_change")
        ].sum()),
    )
    event["move_variance"] = (
        event["expected_square_move"] - event["expected_move"].pow(2)
    ).clip(lower=0)
    event = event.sort_values("date")
    event["expected_move_change"] = event["expected_move"].diff()
    event["max_probability_change"] = event["max_probability"].diff()
    return event



### Meaning of event features

- `expected_move`: probability-weighted signed rate change.
- `expected_abs_move`: expected size of the move, ignoring direction.
- `move_variance`: uncertainty about the possible rate move.
- `entropy`: uncertainty in the whole probability distribution.
- `max_probability`: confidence in the most likely outcome.
- `probability_increase`, `probability_decrease`, `probability_no_change`: separate probability groups.
- `expected_move_change` and `max_probability_change`: changes in market expectations.
- `horizon_month`: months until the meeting.
- `outcome_count`: number of available outcomes.


In [4]:
def add_market_features(market):
    """Create history inputs and future targets.

    History features are shifted by one trading day. Thus a prediction at the
    end of day t cannot accidentally use the return from day t as an input.
    Targets begin on the next trading day.
    """
    frame = market.sort_values("date").copy()
    returns = frame["return_1d"]
    frame["history_return_1d"] = returns.shift(1)
    frame["history_return_5d"] = returns.shift(1).rolling(5).mean()
    frame["history_return_20d"] = returns.shift(1).rolling(20).mean()
    frame["history_variance_5d"] = returns.shift(1).pow(2).rolling(5).mean()
    frame["history_variance_20d"] = returns.shift(1).pow(2).rolling(20).mean()

    for horizon in HORIZONS:
        future_returns = [
            returns.shift(-step) for step in range(1, horizon + 1)
        ]
        frame[f"target_return_{horizon}d"] = (
            frame["close"].shift(-horizon) / frame["close"] - 1
        )
        frame[f"target_variance_{horizon}d"] = sum(
            value.pow(2) for value in future_returns
        )
    return frame


def build_model_data(market, probability):
    event = extract_event_features(probability)
    frame = add_market_features(market)
    return pd.merge_asof(
        frame.sort_values("date"),
        event.sort_values("date"),
        on="date",
        direction="backward",
    ).dropna(subset=["expected_move"])



The SPY history features use only past returns. The target starts on the next trading day: cumulative future return for return models, and the sum of future squared returns for variance models.


## 3. Analyze correlations

Spearman correlation checks whether a feature and future return tend to move together. It is exploratory only; it does not prove that a feature can predict unseen data. The held-out model comparison is more important.


In [5]:
# ============================================================================
# 3. ANALYZE CORRELATIONS
# ============================================================================

EVENT_FEATURES = [
    "expected_move",
    "expected_abs_move",
    "move_variance",
    "entropy",
    "max_probability",
    "probability_increase",
    "probability_decrease",
    "probability_no_change",
    "expected_move_change",
    "max_probability_change",
    "horizon_month",
    "outcome_count",
]
HISTORY_FEATURES = [
    "history_return_1d",
    "history_return_5d",
    "history_return_20d",
    "history_variance_5d",
    "history_variance_20d",
]
ALL_FEATURES = HISTORY_FEATURES + EVENT_FEATURES


def analyze_correlations(model_data):
    """Calculate exploratory Spearman feature/target correlations."""
    targets = [
        f"target_return_{horizon}d" for horizon in HORIZONS
    ]
    correlations = model_data[EVENT_FEATURES + targets].corr(method="spearman")
    return correlations.loc[EVENT_FEATURES, targets]



## 4. Fit models

We use standardized Ridge regression. It is simple and interpretable, and it shrinks unstable coefficients when probability features are correlated. The event-only return model tests whether event probabilities have signal without SPY history. Variance models use QLIKE as an additional metric, but ordinary Ridge can still predict negative variance.


In [6]:
# ============================================================================
# 4. FIT MODELS
# ============================================================================

def qlike(actual, predicted):
    """QLIKE for non-negative variance; predictions are floored for scoring."""
    actual = np.maximum(actual, 1e-8)
    predicted = np.maximum(predicted, 1e-8)
    return np.mean(actual / predicted - np.log(actual / predicted) - 1)


def fit_model(train, test, features, target, is_variance):
    """Fit standardized Ridge and return held-out metrics."""
    train = train.dropna(subset=features + [target])
    test = test.dropna(subset=features + [target])
    estimator = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
    estimator.fit(train[features], train[target])
    prediction = estimator.predict(test[features])
    actual = test[target].to_numpy()
    result = {
        "mae": mean_absolute_error(actual, prediction),
        "rmse": mean_squared_error(actual, prediction) ** 0.5,
        "r2": r2_score(actual, prediction),
        "negative_prediction_rate": np.mean(prediction < 0),
        "rows": len(test),
    }
    if is_variance:
        result["qlike"] = qlike(actual, prediction)
    return result


def fit_all_models(model_data):
    """Fit history-only, event-only, and combined models.

    Event-only return is included to answer whether event probabilities have
    predictive value by themselves, without recent SPY history.
    """
    results = []
    model_features = {
        "history_only": HISTORY_FEATURES,
        "event_only": EVENT_FEATURES,
        "history_plus_event": ALL_FEATURES,
    }
    for horizon in HORIZONS:
        usable = model_data.dropna(
            subset=[
                f"target_return_{horizon}d",
                f"target_variance_{horizon}d",
            ]
        ).iloc[::horizon].reset_index(drop=True)
        split = int(len(usable) * 0.70)
        train, test = usable.iloc[:split], usable.iloc[split:]
        for target_kind, is_variance in [("return", False), ("variance", True)]:
            target = f"target_{target_kind}_{horizon}d"
            candidates = model_features.items()
            if target_kind == "variance":
                candidates = [
                    ("history_only", HISTORY_FEATURES),
                    ("history_plus_event", ALL_FEATURES),
                ]
            for model_name, features in candidates:
                metrics = fit_model(
                    train, test, features, target, is_variance
                )
                results.append(
                    {
                        "horizon": horizon,
                        "target": target_kind,
                        "model": model_name,
                        **metrics,
                    }
                )
    return pd.DataFrame(results)




## 5. Model comparisons

The code below uses the later 30% chronologically as a holdout. Forecast origins are sampled every *H* trading days so multi-day targets do not overlap. Models must be compared only at the same target and horizon.


In [7]:
def main():
    market, probability = load_data()
    model_data = build_model_data(market, probability)
    correlations = analyze_correlations(model_data)
    results = fit_all_models(model_data)
    correlations.to_csv(CORRELATIONS_PATH)
    results.to_csv(RESULTS_PATH, index=False)

    print("Stage 1 - data")
    print(f"SPY rows: {len(market)}")
    print(f"Probability rows: {len(probability)}")
    print(f"Usable modelling rows: {len(model_data)}")
    print("\nStage 3 - event feature / future return Spearman correlations")
    print(correlations.round(4).to_string())
    print("\nStage 5 - held-out model comparisons")
    print(results.round(6).to_string(index=False))
    print("\nHow to read these metrics:")
    print("- MAE and RMSE measure prediction error; lower is better.")
    print("- RMSE penalizes very bad predictions more heavily than MAE.")
    print("- R2 compares with a constant-average prediction; negative means worse.")
    print("- QLIKE is for variance forecasts; lower is better.")
    print("- Negative predictions are acceptable for returns, but not variance.")
    print("- Compare only the same target and horizon; 20-day results have few rows.")
    print("\nImplication:")
    print("- History-only was best for 1-day and 5-day return RMSE.")
    print("- Event-only was slightly best for 20-day return RMSE, but had only 15 test rows.")
    print("- History-plus-event did not beat history-only for variance.")
    print("- These results do not yet prove event features add forecasting value.")
    print("- More independent events, walk-forward tests, and smart-wallet features are needed.")
    print(f"\nSaved: {RESULTS_PATH}")
    print(f"Saved: {CORRELATIONS_PATH}")


if __name__ == "__main__":
    main()


Stage 1 - data
SPY rows: 1184
Probability rows: 9530
Usable modelling rows: 954

Stage 3 - event feature / future return Spearman correlations
                        target_return_1d  target_return_5d  target_return_20d
expected_move                    -0.0294           -0.0249            -0.0289
expected_abs_move                 0.0588            0.1337             0.1459
move_variance                     0.0396            0.1383             0.2531
entropy                           0.0328            0.1238             0.2272
max_probability                  -0.0328           -0.1236            -0.2147
probability_increase              0.0158            0.0863             0.1565
probability_decrease              0.0590            0.0935             0.1075
probability_no_change            -0.0561           -0.1223            -0.1105
expected_move_change             -0.0222            0.0032            -0.0430
max_probability_change           -0.0114            0.0035            -0.0709

### Metrics and implications

- **MAE:** average absolute error; lower is better.
- **RMSE:** error that penalizes very large mistakes more; lower is better.
- **R?:** comparison with a constant-average prediction. Zero means no improvement; negative means worse than that reference.
- **QLIKE:** variance-specific loss; lower is better.
- **Negative prediction rate:** acceptable for returns, but a warning for variance because variance cannot be negative.
- **Rows:** held-out cases; the 20-day test has very few cases, so it is uncertain.

The current implication is that history-only is best for 1-day and 5-day return RMSE. Event-only is slightly best for 20-day return RMSE, but has only 15 test rows. History-plus-event does not beat history-only for variance. This does not prove event features are useless; this small, event-clustered dataset has not yet shown reliable incremental value. More independent events, walk-forward testing, and smart-wallet features are needed.
